# 03 — Corrective RAG (CRAG)

*Level 4 — Adaptive RAG*

## Objective
Grade retrieved evidence *before* trusting it — brute-force search always returns its Top-K, whether or not any of it is actually relevant (see Level 1's `theory/vector_search.md`). CRAG catches that instead of generating from weak evidence anyway.


In [1]:
import sys
from pathlib import Path

LEVEL_DIR = Path.cwd().parent
for sub in ["", "corrective-rag"]:
    sys.path.insert(0, str(LEVEL_DIR / sub) if sub else str(LEVEL_DIR))


In [2]:
from adaptive_common.dataset import prepare
from adaptive_common.retrieval import DenseRetriever
from crag import corrective_retrieve

data = prepare()
retriever = DenseRetriever.from_corpus(data.corpus)


## A question the corpus can actually answer


In [3]:
q = "Where is Russell Hobbs based?"
graded = corrective_retrieve(q, retriever, data.corpus, top_k=5)
print(f"confidence={graded['confidence']:.2f}  trustworthy={graded['trustworthy']}")
for g in graded["graded_results"]:
    print(f"  {g['grade']:<12} {g['doc_id']}")


confidence=0.20  trustworthy=True
  relevant     Russell Hobbs
  irrelevant   Peter Hobbs (engineer)
  irrelevant   FC Red Bull Salzburg
  irrelevant   Southern Professional Hockey League
  irrelevant   ITV Granada


## A question the corpus can NOT answer (out-of-domain, general knowledge)


In [4]:
q2 = "What is the boiling point of water in Celsius?"
graded2 = corrective_retrieve(q2, retriever, data.corpus, top_k=5)
print(f"confidence={graded2['confidence']:.2f}  trustworthy={graded2['trustworthy']}")
for g in graded2["graded_results"]:
    print(f"  {g['grade']:<12} {g['doc_id']}")


confidence=0.00  trustworthy=False
  irrelevant   Antarctic Circumpolar Current
  irrelevant   Hydrochloride
  irrelevant   Cubic metre
  irrelevant   Laurentian Divide
  irrelevant   Acetic oxalic anhydride


## Real, aggregate agreement: does CRAG's confidence track whether gold evidence was actually retrieved?


In [5]:
sys.path.insert(0, str(LEVEL_DIR / "evaluation"))
from adaptive_eval import evaluate_crag

result = evaluate_crag(data.questions, data.corpus, retriever, n=20)
print(result)


{'n_questions': 20, 'agreement': 0.85}


## What I observed

This result went through a real correction worth knowing about: an earlier version of `grade_passage()` checked grades with plain substring containment (`"relevant" in response`), and since `"relevant"` is a literal substring of `"irrelevant"`, checking it first meant **every "irrelevant" verdict was silently read as "relevant."** That bug made CRAG look artificially perfect (confidence always high, agreement 1.0/20) — fixed now with word-boundary matching, and the honest numbers above are lower and far more informative.

With the fix, a **ratio**-based trustworthy threshold ("at least half the Top-K graded relevant") agreed with whether gold evidence was actually retrieved on **0 of 20** real questions — because a question needing two specific supporting documents out of a Top-5 will always have 3 unavoidable "distractor" retrievals graded irrelevant, capping precision at 40% even when retrieval succeeded perfectly. Switching to "**at least 1** relevant passage found" — a recall-oriented check instead of a precision-oriented one — raised agreement to **17 of 20**. `corrective_retrieve()` now uses that as its default.

Two lessons worth carrying forward: a bug in a grading/scoring function can hide behind results that look *too* good, not just ones that look bad — and a confidence metric has to match the actual shape of the question (multi-fact questions are inherently mixed with distractors; don't grade them like single-fact ones).

## Next

[04 — Self-RAG](./04_self_rag.ipynb)
